In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv


In [2]:
test=pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv')
train=pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv')
sample_submission=pd.read_csv('/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv')

In [3]:
train.columns

Index(['space_id', 'operator_ref', 'operator_platform_age_years',
       'operator_platform_age_months', 'operator_active_years',
       'operator_active_months', 'operator_class', 'operator_certified',
       'operator_portfolio_size', 'operator_has_avatar',
       'operator_id_verified', 'zone', 'borough', 'geo_lat', 'geo_lon',
       'space_class', 'access_type', 'max_seats', 'utility_rooms', 'zones',
       'workstations', 'facilities', 'snapshot_start_date',
       'snapshot_end_date', 'min_term_days', 'max_term_days', 'min_term_floor',
       'min_term_ceiling', 'max_term_floor', 'max_term_ceiling',
       'min_term_avg', 'max_term_avg', 'bookable', 'open_slots_a',
       'open_slots_b', 'open_slots_c', 'open_slots_d', 'review_count',
       'review_count_12m', 'review_count_30d', 'open_slots_e',
       'review_count_prev_yr', 'booked_load', 'first_review_date',
       'last_review_date', 'score_overall', 'score_accuracy',
       'score_condition', 'score_onboarding', 'score_supp

In [ ]:
# ================================================================
# 0. LOAD COMPETITION DATA
# ================================================================

import pandas as pd
import numpy as np

train_df = pd.read_csv(
    '/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv'
)

test_df = pd.read_csv(
    '/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv'
)

TARGET = "price_tier"

# Create target and feature matrices
X = train_df.drop(columns=[TARGET]).copy()
y = train_df[TARGET].copy()
X_test = test_df.copy()

print("Data loaded successfully!")
print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("X:", X.shape)
print("X_test:", X_test.shape)
print("y:", y.shape)

In [ ]:
print("Available required variables:")

for name in ["X", "X_test", "y", "train_df", "test_df",
             "sample_submission"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: Available | Type: {type(obj)}")

        if hasattr(obj, "shape"):
            print("   Shape:", obj.shape)
    else:
        print(f"{name}: NOT AVAILABLE")

In [ ]:

# ================================================================
# EXPERIMENT E: CATBOOST OOF VALIDATION + FINAL KAGGLE SUBMISSION
# TOP 100 FACILITY TOKENS | GPU
# OOF TARGET: APPROXIMATELY 0.729 MACRO F1
# FINAL OUTPUT: submission.csv (ID, price_tier)
# ================================================================

import numpy as np
import pandas as pd
import re
import gc
import warnings

from collections import Counter
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD EXISTING ENGINEERED DATA
# ================================================================

required = ["X", "X_test", "y"]

if not all(v in globals() for v in required):
    raise ValueError("X, X_test and y are required. Run feature engineering first.")

X_train_original = X.copy().reset_index(drop=True)
X_test_original = X_test.copy().reset_index(drop=True)
y_all = pd.Series(y).reset_index(drop=True)

X_train_original.columns = X_train_original.columns.astype(str)
X_test_original.columns = X_test_original.columns.astype(str)

y_all = pd.to_numeric(y_all, errors="raise").astype(int)

assert "facilities" in X_train_original.columns
assert "facilities" in X_test_original.columns
assert "space_id" in X_train_original.columns
assert "space_id" in X_test_original.columns
assert len(X_train_original) == len(y_all)

print("Training shape:", X_train_original.shape)
print("Test shape:", X_test_original.shape)
print("\nTarget distribution:")
print(y_all.value_counts(normalize=True).sort_index())

# ================================================================
# 2. IDENTIFY CATEGORICAL FEATURES
# ================================================================

if "cat_features" in globals() and isinstance(
    cat_features, (list, tuple, set)
):
    categorical_cols = [
        str(c) for c in cat_features
        if str(c) in X_train_original.columns
    ]
else:
    categorical_cols = []

for col in X_train_original.columns:
    if (
        X_train_original[col].dtype == "object"
        or isinstance(X_train_original[col].dtype, pd.CategoricalDtype)
        or str(X_train_original[col].dtype) == "string"
    ):
        if col not in categorical_cols:
            categorical_cols.append(col)

categorical_cols = [
    c for c in categorical_cols
    if c not in ["space_id", "facilities"]
]

# ================================================================
# 3. FACILITY TOKENIZATION
# ================================================================

def tokenize_facilities(value):

    if pd.isna(value):
        return []

    value = str(value).lower().strip()

    tokens = re.split(r"[,;|/+\[\]{}]", value)

    return [
        token.strip()
        for token in tokens
        if token.strip()
        and token.strip() != "__missing__"
    ]


train_facility_tokens = X_train_original["facilities"].map(
    tokenize_facilities
)

test_facility_tokens = X_test_original["facilities"].map(
    tokenize_facilities
)

# Learn top 100 tokens ONLY from training data
token_counts = Counter(
    token
    for row in train_facility_tokens
    for token in row
)

selected_tokens = [
    token for token, count in token_counts.most_common(100)
]

print("\nUnique facility tokens:", len(token_counts))
print("Selected facility tokens:", len(selected_tokens))

# ================================================================
# 4. CREATE EXPERIMENT E FEATURES FOR TRAIN AND TEST
# ================================================================

X_train_final = X_train_original.drop(
    columns=["space_id", "facilities"]
).copy()

X_test_final = X_test_original.drop(
    columns=["space_id", "facilities"]
).copy()

generated_cols = []

for token in selected_tokens:

    col_name = "facility_" + re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        token
    )[:45]

    if (
        col_name in X_train_final.columns
        or col_name in generated_cols
    ):
        col_name += "_" + str(len(generated_cols))

    X_train_final[col_name] = train_facility_tokens.map(
        lambda row: int(token in row)
    ).astype("int8")

    X_test_final[col_name] = test_facility_tokens.map(
        lambda row: int(token in row)
    ).astype("int8")

    generated_cols.append(col_name)

# Prepare categorical columns consistently
categorical_cols = [
    c for c in categorical_cols
    if c in X_train_final.columns
]

for col in categorical_cols:

    X_train_final[col] = (
        X_train_final[col]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_final[col] = (
        X_test_final[col]
        .fillna("__MISSING__")
        .astype(str)
    )

# Ensure identical feature order
X_test_final = X_test_final[X_train_final.columns]

print("\nFINAL FEATURE INFORMATION")
print("Training shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)
print("Categorical features:", len(categorical_cols))
print("Generated facility features:", len(generated_cols))

# ================================================================
# 5. CATBOOST PARAMETERS
# ================================================================

CAT_PARAMS = {
    "iterations": 3200,
    "learning_rate": 0.04,
    "depth": 7,
    "l2_leaf_reg": 12,
    "random_strength": 0.4,
    "auto_class_weights": "SqrtBalanced",
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.7,
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",
    "task_type": "GPU",
    "devices": "0",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False
}

# ================================================================
# 6. THREE-FOLD STRATIFIED OOF VALIDATION
# ================================================================

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_predictions = np.full(
    len(y_all),
    -1,
    dtype=int
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X_train_final, y_all), 1
):

    print("\n" + "=" * 60)
    print(f"VALIDATION FOLD {fold}/3")
    print("=" * 60)

    X_fold_train = X_train_final.iloc[train_idx]
    X_fold_valid = X_train_final.iloc[valid_idx]

    y_fold_train = y_all.iloc[train_idx]
    y_fold_valid = y_all.iloc[valid_idx]

    fold_model = CatBoostClassifier(**CAT_PARAMS)

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=categorical_cols,
        verbose=False
    )

    fold_pred = fold_model.predict(
        X_fold_valid
    ).ravel()

    fold_pred = pd.to_numeric(
        pd.Series(fold_pred),
        errors="raise"
    ).astype(int).to_numpy()

    oof_predictions[valid_idx] = fold_pred

    fold_f1 = f1_score(
        y_fold_valid,
        fold_pred,
        average="macro"
    )

    fold_scores.append(fold_f1)

    print(f"Fold {fold} Macro F1: {fold_f1:.7f}")

    del fold_model
    del X_fold_train, X_fold_valid
    del y_fold_train, y_fold_valid

    gc.collect()

# ================================================================
# 7. FINAL OOF SCORE
# ================================================================

assert np.all(oof_predictions != -1)

oof_score = f1_score(
    y_all.to_numpy(),
    oof_predictions,
    average="macro"
)

print("\n" + "=" * 65)
print("EXPERIMENT E — OOF VALIDATION RESULTS")
print("=" * 65)

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i} Macro F1: {score:.7f}")

print("-" * 65)
print("Mean Fold Macro F1:", round(np.mean(fold_scores), 7))
print("Overall OOF Macro F1:", round(oof_score, 7))
print("Previous Experiment E OOF:", 0.7298922)

print("\nClassification Report:")
print(
    classification_report(
        y_all,
        oof_predictions,
        zero_division=0
    )
)

# Save OOF predictions separately
oof_output = pd.DataFrame({
    "actual": y_all,
    "oof_prediction": oof_predictions
})

oof_output.to_csv(
    "experiment_E_oof_predictions.csv",
    index=False
)

print("\nSaved OOF diagnostics: experiment_E_oof_predictions.csv")

# ================================================================
# 8. TRAIN FINAL MODEL ON 100% OF TRAINING DATA
# ================================================================

print("\n" + "=" * 65)
print("TRAINING FINAL CATBOOST MODEL ON FULL TRAINING DATA")
print("=" * 65)

final_model = CatBoostClassifier(**CAT_PARAMS)

final_model.fit(
    X_train_final,
    y_all,
    cat_features=categorical_cols,
    verbose=200
)

print("\nFinal model training completed.")

# ================================================================
# 9. PREDICT ON ACTUAL COMPETITION TEST DATA
# ================================================================

print("\nPredicting on X_test...")

test_predictions = final_model.predict(
    X_test_final
).ravel()

test_predictions = pd.to_numeric(
    pd.Series(test_predictions),
    errors="raise"
).astype(int).to_numpy()

print("Test predictions generated:", len(test_predictions))

# ================================================================
# 10. GET THE REAL COMPETITION ID
# ================================================================

if "sample_submission" in globals():

    if "ID" not in sample_submission.columns:
        raise ValueError("sample_submission does not contain an ID column.")

    if len(sample_submission) != len(test_predictions):
        raise ValueError("sample_submission row count does not match X_test.")

    test_ids = sample_submission["ID"].to_numpy()

elif "test_df" in globals() and "ID" in test_df.columns:

    test_ids = test_df["ID"].to_numpy()

elif "ID" in X_test_original.columns:

    test_ids = X_test_original["ID"].to_numpy()

else:

    raise ValueError(
        "Actual competition ID not found. "
        "Provide sample_submission['ID'], test_df['ID'], "
        "or X_test['ID']. Do not substitute space_id."
    )

if len(test_ids) != len(test_predictions):
    raise ValueError("ID count and prediction count do not match.")

# ================================================================
# 11. CREATE FINAL KAGGLE SUBMISSION
# ================================================================

submission = pd.DataFrame({
    "ID": test_ids,
    "price_tier": test_predictions
})

# Verify required format
assert submission.columns.tolist() == ["ID", "price_tier"]
assert len(submission) == len(X_test_original)
assert submission["ID"].notna().all()
assert submission["price_tier"].notna().all()

submission.to_csv(
    "submission.csv",
    index=False
)

# Save final model
final_model.save_model(
    "catboost_facilities_top100_final.cbm"
)

# ================================================================
# 12. FINAL SUBMISSION CHECKS
# ================================================================

print("\n" + "=" * 65)
print("FINAL KAGGLE SUBMISSION GENERATED")
print("=" * 65)

print("\nSubmission shape:", submission.shape)
print("Submission columns:", submission.columns.tolist())

print("\nFirst 10 rows:")
print(submission.head(10))

print("\nMissing values:")
print(submission.isnull().sum())

print("\nPredicted price_tier distribution:")
print(
    submission["price_tier"]
    .value_counts()
    .sort_index()
)

print("\nPredicted class proportions:")
print(
    submission["price_tier"]
    .value_counts(normalize=True)
    .sort_index()
    .round(5)
)

print("\nFiles saved:")
print("1. submission.csv  <-- SUBMIT THIS TO KAGGLE")
print("2. experiment_E_oof_predictions.csv  <-- VALIDATION ONLY")
print("3. catboost_facilities_top100_final.cbm  <-- TRAINED MODEL")


In [ ]:
sub=pd.read_csv('/kaggle/working/submission.csv')
sub.head()